In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

months   = pd.date_range("2024-01-01", "2025-12-01", freq="MS")
regions  = {"North America":1.0, "EMEA":0.8, "APAC":0.6, "LATAM":0.35}
products = {"Core Platform":(120,0.78), "Analytics Add-on":(60,0.85), "Services":(200,0.45)}  # (avg price, gross margin)
depts    = {"Sales & Marketing":0.18, "R&D":0.14, "G&A":0.08}   # opex as % of planned revenue

tx, bud = [], []
for i, m in enumerate(months):
    season = 1 + 0.15*np.sin(2*np.pi*(m.month-3)/12)
    growth = 1.02**i
    for reg, rw in regions.items():
        for prod, (price, gm) in products.items():
            n_plan = int(2300*rw*season*growth)
            # Planted story: APAC Services misses plan from Jul 2025
            drift = 0.75 if (reg=="APAC" and prod=="Services" and m >= pd.Timestamp("2025-07-01")) else 1.0
            n = int(n_plan*drift*rng.normal(1, 0.03))
            p = rng.normal(price, price*0.1, n).clip(10)
            tx.append(pd.DataFrame({
                "order_date": m + pd.to_timedelta(rng.integers(0, 28, n), unit="D"),
                "region": reg, "product": prod,
                "customer_id": rng.integers(1, 60000, n),
                "revenue": p.round(2),
                "cogs": (p*(1-gm)*rng.normal(1, 0.05, n)).round(2)}))
            bud.append({"month": m, "region": reg, "product": prod,
                        "budget_revenue": round(n_plan*price, 2),
                        "budget_cogs": round(n_plan*price*(1-gm), 2)})

tx, bud = pd.concat(tx), pd.DataFrame(bud)

# Opex: budget = % of planned revenue; planted story: EMEA Sales & Marketing overspend from Apr 2025
plan = bud.groupby(["month","region"]).budget_revenue.sum().reset_index()
op = plan.merge(pd.DataFrame(list(depts.items()), columns=["department","pct"]), how="cross")
op["budget_opex"] = (op.budget_revenue*op.pct).round(2)
over = np.where((op.region=="EMEA")&(op.department=="Sales & Marketing")&(op.month>="2025-04-01"), 1.18, 1.0)
op["actual_opex"] = (op.budget_opex*over*rng.normal(1, 0.03, len(op))).round(2)

tx.to_csv("transactions.csv", index=False)
bud.to_csv("budget_revenue_cogs.csv", index=False)
op[["month","region","department","budget_opex","actual_opex"]].to_csv("opex.csv", index=False)
print(len(tx), "transaction rows")

573250 transaction rows


In [2]:
import os
print([f for f in os.listdir() if f.endswith(".csv")])

['opex.csv', 'budget_revenue_cogs.csv', 'transactions.csv']


In [3]:
# load data into SQL 
import sqlite3

con = sqlite3.connect("fpa.db")
pd.read_csv("transactions.csv").to_sql("transactions", con, if_exists="replace", index=False)
pd.read_csv("budget_revenue_cogs.csv").to_sql("budget", con, if_exists="replace", index=False)
pd.read_csv("opex.csv").to_sql("opex", con, if_exists="replace", index=False)
print("Loaded")

Loaded


In [4]:
# 1. Row count: SQL vs. Python
sql_rows = pd.read_sql("SELECT COUNT(*) AS n FROM transactions", con).iloc[0,0]
print("SQL rows:", sql_rows, "| Python rows:", len(tx))

SQL rows: 573250 | Python rows: 573250


In [5]:
# 2. Total revenue: SQL vs. Python
sql_rev = pd.read_sql("SELECT SUM(revenue) AS r FROM transactions", con).iloc[0,0]
print("SQL revenue:", round(sql_rev, 2), "| Python revenue:", round(tx["revenue"].sum(), 2))

SQL revenue: 72430704.84 | Python revenue: 72430704.84


In [6]:

# 3. No NULLs in key columns
pd.read_sql("""SELECT SUM(revenue IS NULL) AS null_revenue,
                      SUM(cogs IS NULL)    AS null_cogs,
                      SUM(region IS NULL)  AS null_region
               FROM transactions""", con)

,null_revenue,null_cogs,null_region
0,0,0,0
